# Do the 3 code-intelligence tools work?

The ca5 pack declares 9 tools. Across **258 task-runs (129 E4B + 129 31B)** the three
graph tools were called **zero times**, and `system.md` never mentions them.

This probe decides whether that is because they are ignored, or because they are **dead**
(their default backing store is a Google-internal CNS path).

CPU only. No model, no GPU, no task evaluation.


In [ ]:

import glob, os, shutil, subprocess, sys
from pathlib import Path
print("listing /kaggle/input:", sorted(os.listdir("/kaggle/input")))
CANDIDATES = [
    Path("/kaggle/input/datasets/metric/gemma-4-developer-agent-wheelhouse"),
    Path("/kaggle/input/gemma-4-developer-agent-wheelhouse"),
]
WH = next((p for p in CANDIDATES if p.is_dir()), None) or Path(
    (glob.glob("/kaggle/input/**/gemma-4-developer-agent-wheelhouse", recursive=True) or [""])[0])
assert WH and WH.is_dir(), "wheelhouse not mounted"
print("WHEELHOUSE:", WH)
for pat in ("/usr/local/lib/python*/dist-packages/*cutlass*.pth",
            "/usr/local/lib/python*/site-packages/*cutlass*.pth"):
    for p in glob.glob(pat):
        try: os.unlink(p)
        except OSError: pass
tmp = Path("/tmp/wh"); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
for w in WH.glob("*.whl"):
    if "cutlass" in w.name.lower(): continue
    shutil.copy2(w, tmp / w.name.replace("cu128", "+cu128"))
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps",
                    "--force-reinstall"] + [str(p) for p in sorted(tmp.glob("*.whl"))],
                   capture_output=True, text=True)
print("pip rc:", r.returncode, (r.stderr or "")[-400:])
import swegemma, adk_submission
print("IMPORTS OK: adk_submission", getattr(adk_submission, "__version__", "?"),
      "| swegemma", getattr(swegemma, "__version__", "?"))


In [ ]:

import inspect, json, os, traceback
from pathlib import Path

print("\n" + "=" * 70)
print("1. WHERE WOULD THE GRAPH COME FROM?")
print("=" * 70)
try:
    from swegemma.graph import embedding_utils as EU
    for name in [n for n in dir(EU) if "SSTABLE" in n or "DEFAULT" in n]:
        v = getattr(EU, name)
        if isinstance(v, str):
            exists = Path(v).exists()
            print("  %-34s %s  (exists=%s)" % (name, v[:70], exists))
except Exception as e:
    print("  cannot read embedding_utils:", type(e).__name__, e)

print("\n  graph-like files under /kaggle/input:")
hits = 0
seen_top = set()
for root, dirs, files in os.walk("/kaggle/input"):
    depth = root.count("/")
    parts = root.split("/")
    if len(parts) > 6:
        seen_top.add("/".join(parts[:6]))
    if depth > 8:
        dirs[:] = []
        continue
    for f in files:
        if any(k in f.lower() for k in ("graph", ".sst", "embedding", "topolog", "callgraph")):
            print("   ", os.path.join(root, f)[:130])
            hits += 1
            if hits > 15: break
    if hits > 15: break
print("  total graph-ish files found:", hits)
print("  /kaggle/input top-level:", sorted(os.listdir("/kaggle/input")))
for d in ("competitions", "datasets", "models"):
    p = "/kaggle/input/" + d
    if os.path.isdir(p):
        for sub in sorted(os.listdir(p))[:6]:
            print("    %s/%s -> %s" % (d, sub, sorted(os.listdir(os.path.join(p, sub)))[:8]))

print("\n" + "=" * 70)
print("2. DO THE TOOLS RETURN DATA OR ERRORS?")
print("=" * 70)


class Ctx:
    """Minimal stand-in for the harness context the tools expect."""
    graph_dir = None
    repo = "Textualize/rich"
    task = {"repo": "Textualize/rich", "instance_id": "rich_4077"}

    def __getattr__(self, k):
        return None


CASES = [
    ("get_code_neighbors", {"node": "FileProxy"}),
    ("get_code_neighbors", {"node": "FileProxy.isatty"}),
    ("search_similar_code", {"query": "proxy isatty"}),
    ("get_code_subgraph", {"nodes": ["FileProxy"]}),
]
try:
    from swegemma.tools import graph as G
    for fn, kw in CASES:
        f = getattr(G, fn, None)
        if f is None:
            print("  %s: NOT PRESENT" % fn)
            continue
        try:
            out = f(Ctx(), **kw)
            s = out if isinstance(out, str) else json.dumps(out, default=str)
            print("  %-22s %-34s -> %s" % (fn, kw, s[:220]))
        except Exception as e:
            print("  %-22s %-34s -> RAISED %s: %s" % (fn, kw, type(e).__name__, str(e)[:160]))
except Exception:
    print("  cannot import swegemma.tools.graph:")
    traceback.print_exc(limit=4)

print("\n" + "=" * 70)
print("3. ARE THEY REGISTERED IN THE TOOL REGISTRY (i.e. callable at all)?")
print("=" * 70)
try:
    from adk_submission import ToolRegistry
    names = [n for n in dir(ToolRegistry) if not n.startswith("_")]
    print("  ToolRegistry members:", names[:12])
except Exception as e:
    print("  ToolRegistry introspect failed:", type(e).__name__, e)

try:
    from swegemma.tools import __all__ as tools_all
    print("  swegemma.tools exports:", [t for t in tools_all if "code" in t.lower() or "search" in t.lower()])
except Exception as e:
    print("  no __all__:", type(e).__name__)

print("\nPROBE DONE")
